Giai đoạn 4.3: Hybrid

Ý tưởng công thức: hybrid_score = α * CB_score + β * CF_score (α + β = 1)

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movies = pd.read_csv('../data/processed/movies_features.csv')

with open('../models_artifacts/tfidf_matrix.pkl', 'rb') as f:
    tfidf_matrix = pickle.load(f)
with open('../models_artifacts/content_based_candidate_indices.pkl', 'rb') as f:
    candidate_indices = pickle.load(f)
with open('../models_artifacts/user_item_matrix.pkl', 'rb') as f:
    user_item_matrix = pickle.load(f)
with open('../models_artifacts/user_knn_model.pkl', 'rb') as f:
    user_knn = pickle.load(f)

movies_id_to_pos = pd.Series(movies.index, index=movies['id'])
candidate_tmdb_ids = movies.iloc[candidate_indices]['id'].values

print("Candidate pool:", len(candidate_indices))
print("User-Item matrix:", user_item_matrix.shape)

Candidate pool: 22915
User-Item matrix: (671, 3493)


Bước 4.3.1. Hàm gợi ý Hybrid

In [3]:
def normalize(arr):
    if arr.max() - arr.min() < 1e-9:
        return np.zeros_like(arr)
    return (arr - arr.min()) / (arr.max() - arr.min())


def get_hybrid_recommendations(user_id, top_n=5, k_neighbors=10):
    has_cf = user_id in user_item_matrix.index

    if has_cf:
        user_ratings = user_item_matrix.loc[user_id]
        liked_tmdb_ids = user_ratings[user_ratings >= 4].index.tolist()
        n_ratings = (user_ratings > 0).sum()
    else:
        print(f"User {user_id} không có trong CF matrix — dùng 100% Content-Based (cold-start)")
        liked_tmdb_ids, n_ratings = [], 0

    if len(liked_tmdb_ids) == 0 and not has_cf:
        print("Không đủ dữ liệu để gợi ý (cold-start hoàn toàn, chưa có phim đã thích)")
        return pd.DataFrame(), None, None

    # --- CB score: trung bình similarity với các phim đã thích ---
    liked_positions = movies_id_to_pos.reindex(liked_tmdb_ids).dropna().astype(int).values
    if len(liked_positions) > 0:
        sim_matrix = cosine_similarity(tfidf_matrix[liked_positions], tfidf_matrix[candidate_indices])
        cb_scores_raw = sim_matrix.mean(axis=0)
    else:
        cb_scores_raw = np.zeros(len(candidate_indices))

    # --- CF score: tái sử dụng logic Bước 4.2 ---
    if has_cf:
        user_vector = user_item_matrix.loc[user_id].values.reshape(1, -1)
        distances, idx_knn = user_knn.kneighbors(user_vector, n_neighbors=k_neighbors + 1)
        sim_users_pos = idx_knn.flatten()[1:]
        sims = 1 - distances.flatten()[1:]
        weighted_scores = user_item_matrix.iloc[sim_users_pos].T.dot(sims) / (sims.sum() + 1e-9)
        cf_scores_raw = weighted_scores.reindex(candidate_tmdb_ids).fillna(0).values
    else:
        cf_scores_raw = np.zeros(len(candidate_indices))

    cb_norm = normalize(cb_scores_raw)
    cf_norm = normalize(cf_scores_raw)

    # --- Trọng số động theo lượng dữ liệu user ---
    if has_cf:
        alpha = max(0.2, 1 - n_ratings / 50)  # càng nhiều rating, alpha (CB) càng giảm
        beta = 1 - alpha
    else:
        alpha, beta = 1.0, 0.0

    hybrid_scores = alpha * cb_norm + beta * cf_norm

    already_rated_ids = set(user_item_matrix.columns[user_item_matrix.loc[user_id] > 0]) if has_cf else set()

    scores_df = pd.DataFrame({
        'id': candidate_tmdb_ids, 'hybrid_score': hybrid_scores,
        'cb_score': cb_norm, 'cf_score': cf_norm
    })
    scores_df = scores_df[~scores_df['id'].isin(already_rated_ids)]

    top = scores_df.sort_values('hybrid_score', ascending=False).head(top_n)
    result = movies[movies['id'].isin(top['id'])][['id', 'title', 'weighted_rating']].merge(top, on='id')
    result = result.sort_values('hybrid_score', ascending=False).reset_index(drop=True)
    return result, alpha, beta

Bước 4.3.2. Kiểm định với user có nhiều/ít dữ liệu khác nhau

In [4]:
for uid in [1, 5, 300]:
    n_ratings = (user_item_matrix.loc[uid] > 0).sum() if uid in user_item_matrix.index else 0
    print(f"--- Phim dành riêng cho bạn (user_id={uid}, số rating={n_ratings}) ---")
    res, alpha, beta = get_hybrid_recommendations(uid, top_n=5)
    if not res.empty:
        print(f"(alpha={alpha:.2f}, beta={beta:.2f})")
        print(res[['title', 'hybrid_score', 'cb_score', 'cf_score']].to_string(index=False))
    print()

--- Phim dành riêng cho bạn (user_id=1, số rating=20) ---
(alpha=0.60, beta=0.40)
                 title  hybrid_score  cb_score  cf_score
     Beverly Hills Cop      0.480250  0.146598  0.980728
The Godfather: Part II      0.388847  0.095567  0.828767
         The Godfather      0.354856  0.038915  0.828767
            Moonstruck      0.305821  0.068534  0.661752
          Pulp Fiction      0.300591  0.162445  0.507811

--- Phim dành riêng cho bạn (user_id=5, số rating=100) ---
(alpha=0.20, beta=0.80)
                                            title  hybrid_score  cb_score  cf_score
                                  The Incredibles      0.869655  0.348274  1.000000
                               The Princess Bride      0.838192  0.461173  0.932447
The Lord of the Rings: The Fellowship of the Ring      0.827601  0.303368  0.958660
                         The Shawshank Redemption      0.802986  0.257310  0.939404
                                        Toy Story      0.793673  0.40566

Bước 4.3.3. Kiểm tra cold-start hoàn toàn

In [5]:
get_hybrid_recommendations(999999, top_n=5)

User 999999 không có trong CF matrix — dùng 100% Content-Based (cold-start)
Không đủ dữ liệu để gợi ý (cold-start hoàn toàn, chưa có phim đã thích)


(Empty DataFrame
 Columns: []
 Index: [],
 None,
 None)

Bước 4.3.4. Đo thời gian chạy

In [6]:
import time
start = time.time()
_ = get_hybrid_recommendations(1, top_n=5)
print(f"Thời gian: {time.time() - start:.3f} giây")

Thời gian: 0.091 giây


In [7]:
with open('../models_artifacts/movies_id_to_pos.pkl', 'wb') as f:
    pickle.dump(movies_id_to_pos, f)
print("Đã lưu artifact Hybrid")

Đã lưu artifact Hybrid
